# Notebook 4 — Panel Regression: Production Network Topology and Monetary Transmission

This notebook tests whether production network concentration (NCI) moderates the
pass-through of ECB rate changes into bank credit standards across euro area countries.

## Specification

```
BLS_tightening(i,t) = α_i + γ_t + β₁·(ΔRATE_t × NCI_i,t-1) + β₂·NCI_i,t-1 + ε(i,t)
```

- `α_i` country fixed effect
- `γ_t` time fixed effect — absorbs common quarterly shocks including the ECB rate level
- `ΔRATE_t × NCI_i,t-1` interaction: does network concentration dampen transmission?
- **β₁ < 0** is the hypothesis: higher NCI → weaker credit tightening response

**Note on identification:** `ΔRATE_t` alone is absorbed by time fixed effects because the ECB
sets the same rate for all countries in each quarter. The interaction term survives because
NCI varies across countries, creating within-time cross-country variation.

## Sample

15 euro area countries, 2003 Q1 – 2022 Q4. Excluded from regression:
Cyprus and Malta (BLS sample of 3–5 banks; one bank = 20–33pp swing),
Luxembourg (financial centre; BLS reflects international wholesale conditions).

**Run Notebooks 1 and 2 first** to generate the merged regression panel.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    from linearmodels.panel import PanelOLS
    HAS_LINEARMODELS = True
except ImportError:
    HAS_LINEARMODELS = False
    print('linearmodels not installed — using statsmodels fallback.')
    print('Install with: pip install linearmodels')

import statsmodels.formula.api as smf

DATA_PROC = Path('../data/processed')
RESULTS   = Path('../results/figures')
TABLES    = Path('../results/tables')
for p in [RESULTS, TABLES]:
    p.mkdir(parents=True, exist_ok=True)

print(f'linearmodels: {HAS_LINEARMODELS}')

---
## 4.1 Load Merged Panel

In [ ]:
panel_path = DATA_PROC / 'merged_regression_panel.parquet'
if not panel_path.exists():
    raise FileNotFoundError('Run Notebooks 1 and 2 first to generate the merged panel.')

panel = pd.read_parquet(panel_path)
print(f'Panel: {panel.shape}')
print(f'Columns: {list(panel.columns)}')
print()
print(panel.describe().round(3))

---
## 4.2 Prepare Regression Variables

In [ ]:
df = panel.copy().reset_index()

if 'quarter' in df.columns:
    df['quarter'] = pd.to_datetime(df['quarter'])
    df['year']    = df['quarter'].dt.year
    df['qnum']    = df['quarter'].dt.year * 4 + df['quarter'].dt.quarter

# Identify variable columns
rate_col = next((c for c in df.columns if 'delta_rate' in c), None)
bls_col  = next((c for c in df.columns if 'tightening' in c), None)
nci_col  = next((c for c in df.columns if 'nci' in c.lower()), None)
hhi_col  = next((c for c in df.columns if 'hhi' in c.lower()), None)

print(f'Rate:  {rate_col}')
print(f'BLS:   {bls_col}')
print(f'NCI:   {nci_col}')
print(f'HHI:   {hhi_col}')

if rate_col: df = df.rename(columns={rate_col: 'delta_rate'})
if bls_col:  df = df.rename(columns={bls_col:  'bls_tightening'})
if nci_col:  df = df.rename(columns={nci_col:  'nci'})
if hhi_col:  df = df.rename(columns={hhi_col:  'centrality_hhi'})

df['rate_x_nci'] = df['delta_rate'] * df['nci']
if 'centrality_hhi' in df.columns:
    df['rate_x_hhi'] = df['delta_rate'] * df['centrality_hhi']

# Restrict to TiVA coverage window
df = df[(df['year'] >= 2003) & (df['year'] <= 2022)]
df_clean = df.dropna(subset=['bls_tightening', 'delta_rate', 'nci'])

print(f'\nClean panel: {len(df_clean):,} obs | '
      f'{df_clean["country"].nunique()} countries | '
      f'{df_clean["year"].nunique()} years')
print()
print(df_clean[['bls_tightening','delta_rate','nci','rate_x_nci']].describe().round(4))

---
## 4.3 Full-Sample Regressions

In [ ]:
results_table = {}

if HAS_LINEARMODELS and 'country' in df_clean.columns:
    df_panel = df_clean.set_index(['country', 'quarter'])

    specs = {
        'Model 1 — NCI only':
            'bls_tightening ~ nci + EntityEffects + TimeEffects',
        'Model 2 — Interaction (main)':
            'bls_tightening ~ nci + rate_x_nci + EntityEffects + TimeEffects',
    }
    if 'centrality_hhi' in df_clean.columns:
        specs['Model 3 — HHI robustness'] = (
            'bls_tightening ~ centrality_hhi + rate_x_hhi + EntityEffects + TimeEffects'
        )

    for name, formula in specs.items():
        try:
            mod = PanelOLS.from_formula(formula, data=df_panel, drop_absorbed=True)
            res = mod.fit(cov_type='clustered', cluster_entity=True)
            results_table[name] = res
            print(f'\n{"="*60}')
            print(name)
            print('='*60)
            print(res.summary.tables[1])
            print(f'R² (within): {res.rsquared_within:.4f}  N = {res.nobs}')
        except Exception as e:
            print(f'{name}: {e}')

else:
    specs = {
        'Model 1 — NCI only':       'bls_tightening ~ nci + C(country) + C(qnum)',
        'Model 2 — Interaction':    'bls_tightening ~ nci + rate_x_nci + C(country) + C(qnum)',
    }
    for name, formula in specs.items():
        try:
            mod = smf.ols(formula, data=df_clean).fit(
                cov_type='cluster', cov_kwds={'groups': df_clean['country']}
            )
            results_table[name] = mod
            key = [p for p in mod.params.index
                   if any(k in p for k in ['nci', 'rate_x', 'Intercept'])]
            print(f'\n{name}')
            print(mod.summary2().tables[1].loc[key].round(4))
            print(f'R²: {mod.rsquared:.4f}  N: {int(mod.nobs)}')
        except Exception as e:
            print(f'{name}: {e}')

---
## 4.4 Tightening Episodes Only

The production network insulation mechanism operates only when the ECB is raising rates. This specification restricts to quarters with positive rate changes.

In [ ]:
df_tight = df_clean[df_clean['delta_rate'] > 0].copy()
print(f'Tightening quarters: {len(df_tight):,} obs | '
      f'{df_tight["country"].nunique()} countries | '
      f'years {df_tight["year"].min()}–{df_tight["year"].max()}')

if HAS_LINEARMODELS:
    try:
        mod = PanelOLS.from_formula(
            'bls_tightening ~ nci + rate_x_nci + EntityEffects + TimeEffects',
            data=df_tight.set_index(['country', 'quarter']),
            drop_absorbed=True
        )
        res = mod.fit(cov_type='clustered', cluster_entity=True)
        results_table['Model 4 — Tightening only'] = res
        print('\nModel 4 — Tightening episodes only:')
        print(res.summary.tables[1])
        print(f'R² (within): {res.rsquared_within:.4f}  N = {res.nobs}')
    except Exception as e:
        print(f'Failed: {e}')
else:
    try:
        mod = smf.ols(
            'bls_tightening ~ nci + rate_x_nci + C(country) + C(qnum)',
            data=df_tight
        ).fit(cov_type='cluster', cov_kwds={'groups': df_tight['country']})
        results_table['Model 4 — Tightening only'] = mod
        key = [p for p in mod.params.index if any(k in p for k in ['nci', 'rate_x'])]
        print(mod.summary2().tables[1].loc[key].round(4))
    except Exception as e:
        print(f'Failed: {e}')

---
## 4.5 Panel Extension to 2023

TiVA data ends in 2022. We extend to 2023–2024 by carrying forward 2022 network topology values. Intra-EA network structure changes by less than 2% per year, making this a defensible approximation. This adds the 2022–2023 tightening cycle (+450bps in 14 months) to the identification sample.

In [ ]:
topo = pd.read_parquet(DATA_PROC / 'country_topology.parquet').reset_index()
topo_2022 = topo[topo['year'] == 2022].copy()
if len(topo_2022) == 0:
    topo_2022 = topo[topo['year'] == topo['year'].max()].copy()

# Carry forward to 2023–2024
ext = pd.concat(
    [topo] + [topo_2022.assign(year=y) for y in [2023, 2024]],
    ignore_index=True
).drop_duplicates(subset=['year','country'], keep='last')
ext.set_index(['year','country']).to_parquet(DATA_PROC / 'country_topology_extended.parquet')
print(f'Extended topology: {len(ext)} obs ({ext["year"].min()}–{ext["year"].max()})')

from transmission_data import construct_transmission_panel
from network_metrics import load_metrics_panel
from panel_regression import merge_network_and_transmission

bls_df = pd.read_parquet(DATA_PROC / 'bls_net_tightening.parquet')
pr     = pd.read_parquet(DATA_PROC / 'ecb_policy_rate.parquet').iloc[:, 0]
bls_df.index = pd.to_datetime(bls_df.index)
pr.index     = pd.to_datetime(pr.index)

trans_ext = construct_transmission_panel(bls_df, pr)
trans_ext = trans_ext.reset_index()
trans_ext['year'] = pd.to_datetime(trans_ext['quarter']).dt.year
trans_ext = trans_ext[trans_ext['year'].between(2003, 2024)]
trans_ext = trans_ext.set_index(['country', 'quarter'])

try:
    metrics = load_metrics_panel(DATA_PROC)
    merged_ext = merge_network_and_transmission(
        metrics, ext.set_index(['year','country']), trans_ext.reset_index()
    )
    merged_ext.to_parquet(DATA_PROC / 'merged_panel_extended.parquet')
    print(f'Extended panel: {len(merged_ext):,} obs')

    df_ext = merged_ext.reset_index()
    df_ext['quarter'] = pd.to_datetime(df_ext['quarter'])
    df_ext['year']    = df_ext['quarter'].dt.year
    nci_c = next((c for c in df_ext.columns if 'nci' in c.lower()), None)
    if nci_c: df_ext = df_ext.rename(columns={nci_c: 'nci'})
    df_ext['rate_x_nci'] = df_ext['delta_rate'] * df_ext['nci']
    df_ext_tight = df_ext.dropna(subset=['bls_tightening','delta_rate','nci'])
    df_ext_tight = df_ext_tight[df_ext_tight['delta_rate'] > 0]
    print(f'Extended tightening: {len(df_ext_tight):,} obs | '
          f'{df_ext_tight["country"].nunique()} countries | '
          f'years {df_ext_tight["year"].min()}–{df_ext_tight["year"].max()}')

    if HAS_LINEARMODELS and len(df_ext_tight) > 30:
        mod = PanelOLS.from_formula(
            'bls_tightening ~ nci + rate_x_nci + EntityEffects + TimeEffects',
            data=df_ext_tight.set_index(['country','quarter']),
            drop_absorbed=True
        )
        res = mod.fit(cov_type='clustered', cluster_entity=True)
        results_table['Model 5 — Extended 2023'] = res
        print('\nModel 5 — Extended to 2023:')
        print(res.summary.tables[1])
        print(f'R² (within): {res.rsquared_within:.4f}  N = {res.nobs}')
except Exception as e:
    import traceback
    print(f'Extension failed: {e}')
    traceback.print_exc()

---
## 4.6 Robustness: Drop Baltic Outliers

Estonia and Latvia report positive transmission sensitivities — banks in these countries loosen rather than tighten credit when the ECB raises rates. This reflects a structural feature: their banking sectors are dominated by subsidiaries of Swedish banks (Swedbank, SEB) whose lending follows Riksbank policy and Nordic credit cycles rather than ECB conditions. Excluding them tests whether the main result is driven by this anomaly.

In [ ]:
df_no_baltic = df_clean[~df_clean['country'].isin({'EE', 'LV'})].copy()
df_nb_tight  = df_no_baltic[df_no_baltic['delta_rate'] > 0].copy()

print(f'Without EE+LV: {len(df_nb_tight):,} tightening obs | '
      f'{df_nb_tight["country"].nunique()} countries')
print(f'Countries: {sorted(df_nb_tight["country"].unique())}')

if HAS_LINEARMODELS and len(df_nb_tight) > 30:
    try:
        mod = PanelOLS.from_formula(
            'bls_tightening ~ nci + rate_x_nci + EntityEffects + TimeEffects',
            data=df_nb_tight.set_index(['country', 'quarter']),
            drop_absorbed=True
        )
        res = mod.fit(cov_type='clustered', cluster_entity=True)
        results_table['Model 6 — No Baltic outliers'] = res
        print('\nModel 6 — Without EE and LV:')
        print(res.summary.tables[1])
        print(f'R² (within): {res.rsquared_within:.4f}  N = {res.nobs}')

        beta = res.params.get('rate_x_nci', float('nan'))
        pval = res.pvalues.get('rate_x_nci', float('nan'))
        print(f'\nβ_interaction = {beta:.3f},  p = {pval:.4f}')
        if pval < 0.05:
            print('Significant at 5% — result is robust to excluding Baltic outliers.')
        elif pval < 0.10:
            print('Significant at 10% — result is marginally robust.')
        else:
            print('Not significant after excluding Baltic outliers.')
    except Exception as e:
        print(f'Failed: {e}')

---
## 4.7 Results Summary

In [ ]:
print('\n' + '='*70)
print('REGRESSION RESULTS SUMMARY')
print('='*70)
print()
print('Specification:')
print('  BLS(i,t) = α_i + γ_t + β₁·(ΔRATE_t × NCI_i,t-1) + β₂·NCI_i + ε')
print()

rows = []
for name, res in results_table.items():
    row = {'Model': name}
    try:
        if HAS_LINEARMODELS:
            for var, label in [('nci','β_NCI'), ('rate_x_nci','β_interaction')]:
                if var in res.params.index:
                    c = res.params[var]
                    p = res.pvalues[var]
                    s = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else ''
                    row[label] = f'{c:.3f}{s}'
            row['R² within'] = f'{res.rsquared_within:.3f}'
            row['N'] = res.nobs
        else:
            for var, label in [('nci','β_NCI'), ('rate_x_nci','β_interaction')]:
                if var in res.params.index:
                    c = res.params[var]
                    p = res.pvalues[var]
                    s = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else ''
                    row[label] = f'{c:.3f}{s}'
            row['R²'] = f'{res.rsquared:.3f}'
            row['N'] = int(res.nobs)
    except Exception:
        pass
    rows.append(row)

if rows:
    summary = pd.DataFrame(rows).set_index('Model')
    print(summary.fillna('—').to_string())
    summary.to_csv(TABLES / 'regression_results.csv')
    print()
    print('* p<0.10  ** p<0.05  *** p<0.01  (standard errors clustered by country)')

print()
print('--- STEP 4 COMPLETE ---')

---
## Interpretation

**Models 1–3 (full sample):** The interaction coefficient is negative in all specifications but not statistically significant. This is expected: 75% of quarters in the panel have zero rate change. When there is no rate change, the interaction term is identically zero and NCI has no variation to explain. Including these quarters dilutes the signal.

**Model 4 (tightening episodes):** The interaction coefficient is −233.7 (p = 0.017), significant at 5% with the correct sign. During ECB tightening cycles, countries with more concentrated production networks show significantly weaker credit tightening. A one standard deviation increase in NCI (≈0.06) corresponds to approximately 14 percentage points less tightening per 100bps ECB hike.

**Model 5 (extended to 2023):** The result weakens (p = 0.22) when the 2022–2023 cycle is added. The 2022–2023 tightening was qualitatively different — +450bps in 14 months, far more aggressive than any prior cycle. The near-uniform tightening across all EA countries in 2022–2023 suggests the insulation mechanism may be nonlinear: present at moderate policy intensities but overwhelmed at extreme shocks.

**Model 6 (no Baltic outliers):** The result is robust (β = −177.9, p = 0.045). Estonia and Latvia show anomalous positive sensitivity due to Scandinavian banking structure; the core finding does not depend on them.